# Table 3: Civil Code Qwen3-8B LoRA (checkpoint-280) 评估

运行两项测试：
1. **Memory Test**: 背诵 1260 条民法典 — 计算 Effective Memory / Invalid Duplicate / Error
2. **Retrieval Test**: 362 道民法典问题 top-5 法条预测 — 计算 P@5 / R@5 / F1@5

AutoDL RTX 5090 16GB | QLoRA 4-bit

In [1]:
# ============================================================
# Cell 1: Config & Imports
# ============================================================
import os, json, re
from tqdm import tqdm
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

# Download text2vec model via HF mirror first
from huggingface_hub import snapshot_download
os.makedirs('/root/autodl-tmp/models', exist_ok=True)
model_dir = snapshot_download('shibing624/text2vec-base-chinese',
                               cache_dir='/root/autodl-tmp/models',
                               endpoint='https://hf-mirror.com')
from sentence_transformers import SentenceTransformer, util
sim_model = SentenceTransformer(model_dir)

# === Config ===
BASE_MODEL = '/root/autodl-tmp/employee_code/base_model'
LORA_PATH = '/root/autodl-tmp/civil_ft/checkpoints/checkpoint-280'
DATA_DIR = '/root/autodl-tmp/civil_ft/data'
OUTPUT_DIR = '/root/autodl-tmp/civil_ft/results_table3'
os.makedirs(OUTPUT_DIR, exist_ok=True)

ARTICLES_FILE = os.path.join(DATA_DIR, 'civil_code_articles_cleaned.json')
EVAL_FILE = os.path.join(DATA_DIR, 'eval_civil_code_questions.json')

print('PyTorch', torch.__version__)
print('CUDA:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A')
print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
print('Similarity model ready:', model_dir)

Fetching 22 files:   0%|          | 0/22 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: /root/autodl-tmp/models/models--shibing624--text2vec-base-chinese/snapshots/183bb99aa7af74355fb58d16edf8c13ae7c5433e
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


PyTorch 2.8.0+cu128
CUDA: NVIDIA GeForce RTX 4080 SUPER
VRAM: 33.8 GB
Similarity model ready: /root/autodl-tmp/models/models--shibing624--text2vec-base-chinese/snapshots/183bb99aa7af74355fb58d16edf8c13ae7c5433e


In [2]:
# ============================================================
# Cell 2: Load Model + LoRA
# ============================================================
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

print('Loading base model...')
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4'
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb,
    device_map='auto',
    trust_remote_code=True
)
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

print('Loading LoRA adapter:', LORA_PATH)
model = PeftModel.from_pretrained(model, LORA_PATH)
model.eval()
print('Model ready.')

Loading base model...


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

Loading LoRA adapter: /root/autodl-tmp/civil_ft/checkpoints/checkpoint-280
Model ready.


In [3]:
# ============================================================
# Cell 3: Generate + Canon helpers (same as original compare_laws.py logic)
# ============================================================
import re

# --- Canonical article name ---
CN_M = {'零':0,'〇':0,'○':0,'一':1,'二':2,'两':2,'三':3,'四':4,'五':5,'六':6,'七':7,'八':8,'九':9}
CN_U = {'十':10,'百':100,'千':1000,'万':10000}

def c2i(t):
    if not t: return None
    t = str(t).strip().translate(str.maketrans('０１２３４５６７８９', '0123456789'))
    if t.isdigit(): return int(t)
    tot = sec = num = 0
    for c in t:
        if c in CN_M: num = CN_M[c]
        elif c in CN_U:
            u = CN_U[c]
            if u == 10000: sec = (sec + (num or 0)) * u; tot += sec; sec = num = 0
            else:
                if num == 0: num = 1
                sec += num * u; num = 0
        else: return None
    return tot + sec + num

def canon(a):
    if not isinstance(a, str): return ''
    x = a.strip(); x = re.sub(r'\s+', '', x); x = x.replace('（', '(').replace('）', ')')
    m = re.match(r'^(《.*?》)第([0-9一二三四五六七八九十百千万零〇○两]+)条$', x)
    if not m: return x
    l = re.sub(r'^《中华人民共和国', '《', m.group(1))
    n = c2i(m.group(2))
    return l + '第' + str(n) + '条' if n is not None else l + '第' + m.group(2) + '条'

# --- Generate using chat template ---
@torch.no_grad()
def generate(prompt, max_new=400):
    ms = [{'role': 'user', 'content': prompt}]
    tx = tokenizer.apply_chat_template(ms, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer(tx, return_tensors='pt').to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=max_new, do_sample=False,
                             eos_token_id=tokenizer.eos_token_id, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()

# --- Memory test thresholds (same as original compare_laws.py) ---
PASS_THRESHOLD = 0.85  # cosine similarity >= 0.85 -> valid memory
FAIL_THRESHOLD = 0.60  # cosine similarity < 0.60 -> fail (informational)

print('Helpers ready.')
print(f'PASS_THRESHOLD={PASS_THRESHOLD}, FAIL_THRESHOLD={FAIL_THRESHOLD}')

Helpers ready.
PASS_THRESHOLD=0.85, FAIL_THRESHOLD=0.6


In [4]:
# ============================================================
# Cell 4: Load Articles + Pre-compute Embeddings
# ============================================================
import os, json

with open(ARTICLES_FILE, 'r', encoding='utf-8') as f:
    raw_articles = json.load(f)

articles = [{'name': canon(a['name']), 'content': a['content'].strip()} for a in raw_articles]
articles = sorted(articles, key=lambda x: x['name'])
print('Unique Civil Code articles:', len(articles))

# Pre-compute standard embeddings (same as compare_laws.py)
standard_texts = [a['content'] for a in articles]
print('Encoding standard articles...')
standard_embeddings = sim_model.encode(standard_texts, convert_to_tensor=True, show_progress_bar=True)
print('Standard embeddings ready:', standard_embeddings.shape)

Unique Civil Code articles: 1260
Encoding standard articles...


Batches:   0%|          | 0/40 [00:00<?, ?it/s]

Standard embeddings ready: torch.Size([1260, 768])


In [5]:
# ============================================================
# Cell 5: Run Memory Test (same logic as compare_laws.py)
# ============================================================
import os, json, re
from tqdm import tqdm

CHECKPOINT_FILE = os.path.join(OUTPUT_DIR, 'memory_test_checkpoint.partial.json')
SAVE_EVERY = 100

memory_results = []
claimed_std_indices = set()  # indices of standard articles already claimed
exact_count = 0
misplaced_count = 0
repetition_count = 0
fail_count = 0
start_idx = 0

if os.path.exists(CHECKPOINT_FILE):
    print('Found partial save, resuming...')
    with open(CHECKPOINT_FILE, 'r', encoding='utf-8') as f:
        saved = json.load(f)
    memory_results = saved['results']
    start_idx = saved['next_idx']
    exact_count = saved['exact_count']
    misplaced_count = saved['misplaced_count']
    repetition_count = saved['repetition_count']
    fail_count = saved['fail_count']
    claimed_std_indices = set(saved['claimed_std_indices'])
    print(f'  Resuming from idx={start_idx} ({start_idx}/{len(articles)} done)')

for idx in range(start_idx, len(articles)):
    art = articles[idx]
    name = art['name']
    prompt = '请背诵' + name + '。'
    raw = generate(prompt, max_new=400)

    # Encode generated text
    gen_embedding = sim_model.encode(raw, convert_to_tensor=True)

    # Current score: compare to the correct article (same index)
    current_score = util.cos_sim(gen_embedding, standard_embeddings[idx]).item()

    # Global best match
    cos_scores = util.cos_sim(gen_embedding, standard_embeddings)[0]
    best_idx = torch.argmax(cos_scores).item()
    best_score = cos_scores[best_idx].item()

    result = {'idx': idx, 'name': name, 'raw': raw[:200],
              'current_score': current_score, 'best_score': best_score, 'best_idx': best_idx}

    # Step 1: exact match (same as original)
    if current_score >= PASS_THRESHOLD:
        claimed_std_indices.add(idx)
        result['status_type'] = 'exact'
        exact_count += 1
    # Step 2: misplaced / repetition / fail
    else:
        if best_score >= PASS_THRESHOLD:
            if best_idx in claimed_std_indices:
                result['status_type'] = 'repetition'
                repetition_count += 1
            else:
                result['status_type'] = 'misplaced'
                misplaced_count += 1
                claimed_std_indices.add(best_idx)
        else:
            result['status_type'] = 'fail'
            fail_count += 1

    memory_results.append(result)

    done = idx + 1
    if done % SAVE_EVERY == 0 or done == len(articles):
        valid = exact_count + misplaced_count
        print(f'  [{done}/{len(articles)}] valid={valid/done:.2%} rep={repetition_count/done:.2%} fail={fail_count/done:.2%}  -> saving...')
        with open(CHECKPOINT_FILE, 'w', encoding='utf-8') as f:
            json.dump({
                'next_idx': done,
                'exact_count': exact_count, 'misplaced_count': misplaced_count,
                'repetition_count': repetition_count, 'fail_count': fail_count,
                'claimed_std_indices': list(claimed_std_indices),
                'results': memory_results
            }, f, ensure_ascii=False)

total = len(articles)
valid = exact_count + misplaced_count
mem_summary = {
    'effective_memory': round(valid / total, 4),
    'invalid_duplicate': round(repetition_count / total, 4),
    'error_hallucination': round(fail_count / total, 4),
    'exact_count': exact_count, 'misplaced_count': misplaced_count,
    'repetition_count': repetition_count, 'fail_count': fail_count,
    'total': total
}

print()
print('=== Memory Test Results ===')
print('Effective Memory (exact+misplaced): {:.2%} ({}/{})'.format(valid/total, valid, total))
print('  - Exact:      {:.2%} ({})'.format(exact_count/total, exact_count))
print('  - Misplaced:  {:.2%} ({})'.format(misplaced_count/total, misplaced_count))
print('Invalid Duplicate:                 {:.2%} ({})'.format(repetition_count/total, repetition_count))
print('Error/Hallucination:               {:.2%} ({})'.format(fail_count/total, fail_count))

with open(os.path.join(OUTPUT_DIR, 'memory_test_checkpoint280.json'), 'w', encoding='utf-8') as f:
    json.dump({'summary': mem_summary, 'per_article': memory_results}, f, ensure_ascii=False, indent=2)
print('Saved to memory_test_checkpoint280.json')
if os.path.exists(CHECKPOINT_FILE):
    os.remove(CHECKPOINT_FILE)

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  [100/1260] valid=63.00% rep=13.00% fail=24.00%  -> saving...
  [200/1260] valid=59.00% rep=20.00% fail=21.00%  -> saving...
  [300/1260] valid=58.00% rep=21.33% fail=20.67%  -> saving...
  [400/1260] valid=61.50% rep=19.25% fail=19.25%  -> saving...
  [500/1260] valid=60.40% rep=19.60% fail=20.00%  -> saving...
  [600/1260] valid=60.33% rep=21.00% fail=18.67%  -> saving...
  [700/1260] valid=59.29% rep=22.57% fail=18.14%  -> saving...
  [800/1260] valid=59.62% rep=22.12% fail=18.25%  -> saving...
  [900/1260] valid=58.89% rep=23.00% fail=18.11%  -> saving...
  [1000/1260] valid=59.60% rep=23.30% fail=17.10%  -> saving...
  [1100/1260] valid=58.18% rep=24.55% fail=17.27%  -> saving...
  [1200/1260] valid=57.08% rep=25.25% fail=17.67%  -> saving...
  [1260/1260] valid=56.03% rep=25.87% fail=18.10%  -> saving...

=== Memory Test Results ===
Effective Memory (exact+misplaced): 56.03% (706/1260)
  - Exact:      33.25% (419)
  - Misplaced:  22.78% (287)
Invalid Duplicate:                 2